In [5]:
# ============================================================
# SECTION 1 — CREATE ECOBOTX-EDGE MODEL YAML
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# MODEL DIRECTORY
# ------------------------------------------------------------

MODEL_DIR = Path(
    r"G:\EcoBotX_Object_Detection"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# MODEL YAML PATH
# ------------------------------------------------------------

MODEL_YAML = MODEL_DIR / "ecobotx_edge.yaml"


# ------------------------------------------------------------
# ECOBOTX-EDGE ARCHITECTURE
# ------------------------------------------------------------

architecture_yaml = """
# ============================================================
# EcoBotX-Edge — Experiment 6
#
# Lightweight edge-oriented YOLOv8 architecture
#
# Design:
#   - GhostConv for downsampling
#   - C3Ghost for lightweight feature extraction
#   - Standard P3/P4/P5 detection
#   - NO CBAM
#   - NO P2 detection head
#
# Target:
#   Raspberry Pi / Jetson
#   RTX 3050 4GB development laptop
# ============================================================

nc: 4

depth_multiple: 0.33
width_multiple: 0.25


# ============================================================
# BACKBONE
# ============================================================

backbone:

  # [from, repeats, module, args]

  - [-1, 1, Conv,      [64, 3, 2]]       # 0 - P1/2
  - [-1, 1, GhostConv, [128, 3, 2]]      # 1 - P2/4
  - [-1, 3, C3Ghost,   [128, True]]      # 2

  - [-1, 1, GhostConv, [256, 3, 2]]      # 3 - P3/8
  - [-1, 6, C3Ghost,   [256, True]]      # 4

  - [-1, 1, GhostConv, [512, 3, 2]]      # 5 - P4/16
  - [-1, 6, C3Ghost,   [512, True]]      # 6

  - [-1, 1, GhostConv, [1024, 3, 2]]     # 7 - P5/32
  - [-1, 3, C3Ghost,   [1024, True]]     # 8

  - [-1, 1, SPPF,      [1024, 5]]        # 9


# ============================================================
# HEAD
# ============================================================

head:

  # ----------------------------------------------------------
  # P5 -> P4
  # ----------------------------------------------------------

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 10

  - [[-1, 6], 1, Concat, [1]]                   # 11

  - [-1, 3, C3Ghost, [512]]                     # 12


  # ----------------------------------------------------------
  # P4 -> P3
  # ----------------------------------------------------------

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 13

  - [[-1, 4], 1, Concat, [1]]                   # 14

  - [-1, 3, C3Ghost, [256]]                     # 15 - P3/8


  # ----------------------------------------------------------
  # P3 -> P4
  # ----------------------------------------------------------

  - [-1, 1, GhostConv, [256, 3, 2]]             # 16

  - [[-1, 12], 1, Concat, [1]]                   # 17

  - [-1, 3, C3Ghost, [512]]                     # 18 - P4/16


  # ----------------------------------------------------------
  # P4 -> P5
  # ----------------------------------------------------------

  - [-1, 1, GhostConv, [512, 3, 2]]             # 19

  - [[-1, 9], 1, Concat, [1]]                    # 20

  - [-1, 3, C3Ghost, [1024]]                     # 21 - P5/32


  # ----------------------------------------------------------
  # DETECTION
  # ----------------------------------------------------------

  - [[15, 18, 21], 1, Detect, [nc]]             # 22
"""


# ------------------------------------------------------------
# WRITE YAML FILE
# ------------------------------------------------------------

with open(
    MODEL_YAML,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        architecture_yaml.strip()
    )


print("=" * 70)
print("EcoBotX-Edge YAML CREATED")
print("=" * 70)

print("\nLocation:")
print(MODEL_YAML)

print("\nExists:")
print(MODEL_YAML.exists())

print("\nFile size:")
print(f"{MODEL_YAML.stat().st_size:,} bytes")

EcoBotX-Edge YAML CREATED

Location:
G:\EcoBotX_Object_Detection\ecobotx_edge.yaml

Exists:
True

File size:
2,853 bytes


In [6]:
# ============================================================
# SECTION 2 — VERIFY MODEL YAML
# ============================================================

print("=" * 70)
print("ECOBOTX-EDGE YAML")
print("=" * 70)

with open(
    MODEL_YAML,
    "r",
    encoding="utf-8"
) as f:

    print(f.read())

ECOBOTX-EDGE YAML
# ============================================================
# EcoBotX-Edge — Experiment 6
#
# Lightweight edge-oriented YOLOv8 architecture
#
# Design:
#   - GhostConv for downsampling
#   - C3Ghost for lightweight feature extraction
#   - Standard P3/P4/P5 detection
#   - NO CBAM
#   - NO P2 detection head
#
# Target:
#   Raspberry Pi / Jetson
#   RTX 3050 4GB development laptop
# ============================================================

nc: 4

depth_multiple: 0.33
width_multiple: 0.25


# ============================================================
# BACKBONE
# ============================================================

backbone:

  # [from, repeats, module, args]

  - [-1, 1, Conv,      [64, 3, 2]]       # 0 - P1/2
  - [-1, 1, GhostConv, [128, 3, 2]]      # 1 - P2/4
  - [-1, 3, C3Ghost,   [128, True]]      # 2

  - [-1, 1, GhostConv, [256, 3, 2]]      # 3 - P3/8
  - [-1, 6, C3Ghost,   [256, True]]      # 4

  - [-1, 1, GhostConv, [512, 3, 2]]      # 5 - P4

In [7]:
# ============================================================
# SECTION 3 — CHECK ULTRALYTICS MODULE SUPPORT
# ============================================================

import ultralytics
from ultralytics import YOLO

print("=" * 70)
print("ULTRALYTICS ENVIRONMENT")
print("=" * 70)

print(
    "Ultralytics version:",
    ultralytics.__version__
)


# ------------------------------------------------------------
# CHECK COMMON MODULES
# ------------------------------------------------------------

try:

    from ultralytics.nn.modules import (
        Conv,
        GhostConv,
        SPPF,
        C3Ghost
    )

    print("\n✓ Conv      available")
    print("✓ GhostConv available")
    print("✓ SPPF      available")
    print("✓ C3Ghost   available")

except ImportError as e:

    print("\n✗ Custom module problem")
    print("\nError:")
    print(e)

    raise

ULTRALYTICS ENVIRONMENT
Ultralytics version: 8.4.126

✓ Conv      available
✓ GhostConv available
✓ SPPF      available
✓ C3Ghost   available


In [8]:
# ============================================================
# SECTION 4 — BUILD ECOBOTX-EDGE
# ============================================================

print("=" * 70)
print("BUILDING ECOBOTX-EDGE")
print("=" * 70)

model = YOLO(
    str(MODEL_YAML)
)

print("\n✓ EcoBotX-Edge model constructed successfully.")

BUILDING ECOBOTX-EDGE

✓ EcoBotX-Edge model constructed successfully.


In [9]:
# ============================================================
# SECTION 5 — MODEL COMPLEXITY
# ============================================================

print("=" * 70)
print("ECOBOTX-EDGE COMPLEXITY")
print("=" * 70)

model.info(
    detailed=True
)

ECOBOTX-EDGE COMPLEXITY
layer                                    name                type  gradient  parameters               shape        mu     sigma
    0                     model.0.conv.weight              Conv2d      True         432       [16, 3, 3, 3]   0.00396     0.111        float32
    1                       model.0.bn.weight         BatchNorm2d      True          16                [16]         1         0        float32
    1                         model.0.bn.bias         BatchNorm2d      True          16                [16]         0         0        float32
    2                             model.0.act                SiLU     False           0                  []         -         -              -
    3                 model.1.cv1.conv.weight              Conv2d      True        2304      [16, 16, 3, 3]  0.000832    0.0479        float32
    4                   model.1.cv1.bn.weight         BatchNorm2d      True          16                [16]         1         0      

(238, 1719744, 1719728, 5.1428864)

In [10]:
# ============================================================
# SECTION 5B — PARAMETER COUNT
# ============================================================

total_params = sum(
    p.numel()
    for p in model.model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.model.parameters()
    if p.requires_grad
)

print(
    f"Total parameters     : {total_params:,}"
)

print(
    f"Trainable parameters : {trainable_params:,}"
)

Total parameters     : 1,719,744
Trainable parameters : 1,719,728


In [11]:
# ============================================================
# SECTION 6 — FORWARD PASS TEST
# ============================================================

import numpy as np

IMGSZ = 416

dummy_image = np.zeros(
    (IMGSZ, IMGSZ, 3),
    dtype=np.uint8
)


print("=" * 70)
print("FORWARD PASS TEST")
print("=" * 70)


try:

    results = model.predict(
        source=dummy_image,
        imgsz=IMGSZ,
        device=0 if __import__("torch").cuda.is_available() else "cpu",
        verbose=False
    )

    print("\n✓ Forward pass successful.")

except Exception as e:

    print("\n✗ Forward pass failed.")

    print("\nError:")
    print(e)

    raise

FORWARD PASS TEST

✓ Forward pass successful.


In [13]:
# ============================================================
# SECTION 7 — DATASET CONFIGURATION CHECK
# ============================================================

from pathlib import Path
import yaml

DATA = Path(
    r"G:\EcoBotX_YOLO\dataset.yaml"
)

print("=" * 70)
print("DATASET CONFIGURATION")
print("=" * 70)


# ------------------------------------------------------------
# CHECK YAML EXISTS
# ------------------------------------------------------------

if not DATA.exists():

    raise FileNotFoundError(
        f"Dataset YAML not found:\n{DATA}"
    )


# ------------------------------------------------------------
# READ YAML
# ------------------------------------------------------------

with open(
    DATA,
    "r",
    encoding="utf-8"
) as f:

    dataset_config = yaml.safe_load(f)


print("\nRaw dataset YAML:")
print(dataset_config)


# ------------------------------------------------------------
# READ CLASS NAMES
# ------------------------------------------------------------

names = dataset_config.get(
    "names",
    None
)


if names is None:

    raise ValueError(
        "The dataset.yaml does not contain a 'names' field."
    )


# ------------------------------------------------------------
# HANDLE DIFFERENT YOLO NAME FORMATS
# ------------------------------------------------------------

if isinstance(names, dict):

    # Example:
    # names:
    #   0: BOTTLE
    #   1: CAN
    #   2: PAPER
    #   3: WRAPPER

    names = {
        int(k): v
        for k, v in names.items()
    }

    class_names = [
        names[i]
        for i in sorted(names.keys())
    ]


elif isinstance(names, list):

    # Example:
    # names:
    #   - BOTTLE
    #   - CAN
    #   - PAPER
    #   - WRAPPER

    class_names = names


else:

    raise ValueError(
        "Unsupported 'names' format in dataset.yaml."
    )


# ------------------------------------------------------------
# DERIVE NUMBER OF CLASSES
# ------------------------------------------------------------

nc = len(class_names)


# ------------------------------------------------------------
# PRINT RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET CLASS INFORMATION")
print("=" * 70)

print(
    f"Number of classes : {nc}"
)

for i, class_name in enumerate(class_names):

    print(
        f"{i}: {class_name}"
    )


# ------------------------------------------------------------
# EXPECTED ECOBOTX CLASSES
# ------------------------------------------------------------

expected_classes = [
    "BOTTLE",
    "CAN",
    "PAPER",
    "WRAPPER"
]


# ------------------------------------------------------------
# VALIDATE NUMBER OF CLASSES
# ------------------------------------------------------------

if nc != 4:

    raise ValueError(
        f"Expected 4 classes, "
        f"but found {nc}: {class_names}"
    )


# ------------------------------------------------------------
# VALIDATE CLASS ORDER
# ------------------------------------------------------------

if class_names != expected_classes:

    print("\nWARNING: Class order is different.")

    print(
        "Expected:",
        expected_classes
    )

    print(
        "Found:",
        class_names
    )

else:

    print(
        "\n✓ Class order verified."
    )


print("\n✓ Dataset configuration check completed.")

DATASET CONFIGURATION

Raw dataset YAML:
{'path': 'G:\\EcoBotX_YOLO', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0: 'BOTTLE', 1: 'CAN', 2: 'PAPER', 3: 'WRAPPER'}}

DATASET CLASS INFORMATION
Number of classes : 4
0: BOTTLE
1: CAN
2: PAPER
3: WRAPPER

✓ Class order verified.

✓ Dataset configuration check completed.


In [14]:
# ============================================================
# SECTION 8 — EXPERIMENT 6 CONFIGURATION
# ============================================================

from pathlib import Path
import torch
import time
import json
import pandas as pd


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

DATA = r"G:\EcoBotX_YOLO\dataset.yaml"

MODEL_YAML = r"G:\EcoBotX_Object_Detection\ecobotx_edge.yaml"

PRETRAINED = r"G:\EcoBotX_Object_Detection\yolov8n.pt"

PROJECT = r"G:\EcoBotX_YOLO_training"

NAME = "experiment6_ecobotx_edge"


# ------------------------------------------------------------
# TRAINING SETTINGS
# ------------------------------------------------------------

IMGSZ = 416

EPOCHS = 100

BATCH = 8

WORKERS = 4

DEVICE = 0 if torch.cuda.is_available() else "cpu"


# ------------------------------------------------------------
# PRETRAINED WEIGHT OPTION
# ------------------------------------------------------------

USE_PRETRAINED = True


# ------------------------------------------------------------
# OUTPUT DIRECTORY
# ------------------------------------------------------------

OUTPUT_DIR = Path(PROJECT) / NAME

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# PRINT CONFIGURATION
# ------------------------------------------------------------

print("=" * 70)
print("ECOBOTX-EDGE — EXPERIMENT 6")
print("=" * 70)

print(f"\nDataset       : {DATA}")
print(f"Architecture  : {MODEL_YAML}")
print(f"Pretrained    : {PRETRAINED}")

print("\nTraining:")
print(f"Image size    : {IMGSZ}")
print(f"Epochs        : {EPOCHS}")
print(f"Batch size    : {BATCH}")
print(f"Workers       : {WORKERS}")
print(f"Device        : {DEVICE}")

print("\nPretrained initialization:", USE_PRETRAINED)

print("\nOutput:")
print(OUTPUT_DIR)

ECOBOTX-EDGE — EXPERIMENT 6

Dataset       : G:\EcoBotX_YOLO\dataset.yaml
Architecture  : G:\EcoBotX_Object_Detection\ecobotx_edge.yaml
Pretrained    : G:\EcoBotX_Object_Detection\yolov8n.pt

Training:
Image size    : 416
Epochs        : 100
Batch size    : 8
Workers       : 4
Device        : 0

Pretrained initialization: True

Output:
G:\EcoBotX_YOLO_training\experiment6_ecobotx_edge


In [15]:
# ============================================================
# SECTION 9 — VERIFY MODEL + DATASET
# ============================================================

print("=" * 70)
print("FILE VERIFICATION")
print("=" * 70)


required_files = {

    "Dataset YAML":
        Path(DATA),

    "Model YAML":
        Path(MODEL_YAML),

    "Pretrained YOLOv8n":
        Path(PRETRAINED)
}


for name, path in required_files.items():

    if path.exists():

        print(
            f"✓ {name}: {path}"
        )

    else:

        if name == "Pretrained YOLOv8n":

            print(
                f"⚠ {name} not found."
            )

            print(
                "Training can continue from scratch."
            )

        else:

            raise FileNotFoundError(
                f"{name} not found:\n{path}"
            )


print("\n✓ Required Experiment 6 files verified.")

FILE VERIFICATION
✓ Dataset YAML: G:\EcoBotX_YOLO\dataset.yaml
✓ Model YAML: G:\EcoBotX_Object_Detection\ecobotx_edge.yaml
✓ Pretrained YOLOv8n: G:\EcoBotX_Object_Detection\yolov8n.pt

✓ Required Experiment 6 files verified.


In [16]:
# ============================================================
# SECTION 10 — BUILD ECOBOTX-EDGE
# ============================================================

from ultralytics import YOLO


print("=" * 70)
print("BUILDING ECOBOTX-EDGE")
print("=" * 70)


model = YOLO(
    MODEL_YAML
)


print("\n✓ EcoBotX-Edge architecture created.")

BUILDING ECOBOTX-EDGE

✓ EcoBotX-Edge architecture created.


In [17]:
# ============================================================
# SECTION 11 — MODEL COMPLEXITY
# ============================================================

print("=" * 70)
print("ECOBOTX-EDGE MODEL COMPLEXITY")
print("=" * 70)


total_params = sum(
    p.numel()
    for p in model.model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in model.model.parameters()
    if p.requires_grad
)


print(
    f"\nTotal parameters     : {total_params:,}"
)

print(
    f"Trainable parameters : {trainable_params:,}"
)


print("\nUltralytics model information:")

model.info(
    detailed=False
)

ECOBOTX-EDGE MODEL COMPLEXITY

Total parameters     : 1,719,744
Trainable parameters : 1,719,728

Ultralytics model information:
ecobotx_edge summary: 238 layers, 1,719,744 parameters, 1,719,728 gradients, 5.1 GFLOPs


(238, 1719744, 1719728, 5.1428864)

In [18]:
# ============================================================
# SECTION 12 — OPTIONAL PRETRAINED INITIALIZATION
# ============================================================

pretrained_path = Path(
    PRETRAINED
)


if USE_PRETRAINED and pretrained_path.exists():

    print("=" * 70)
    print("PRETRAINED INITIALIZATION")
    print("=" * 70)

    print(
        "\nSource:",
        pretrained_path
    )

    try:

        model.load(
            str(pretrained_path)
        )

        print(
            "\n✓ YOLOv8n compatible weights loaded where possible."
        )

    except Exception as e:

        print(
            "\n⚠ Pretrained initialization failed."
        )

        print(
            "Error:",
            e
        )

        print(
            "\nContinuing with the custom architecture."
        )

else:

    print(
        "Training from random initialization."
    )

PRETRAINED INITIALIZATION

Source: G:\EcoBotX_Object_Detection\yolov8n.pt
Transferred 83/559 items from pretrained weights

✓ YOLOv8n compatible weights loaded where possible.


In [19]:
# ============================================================
# SECTION 13 — GPU MEMORY CHECK
# ============================================================

print("=" * 70)
print("GPU MEMORY STATUS")
print("=" * 70)


if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

    gpu_name = torch.cuda.get_device_name(0)

    total_vram = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    allocated = (
        torch.cuda.memory_allocated(0)
        / (1024 ** 3)
    )

    reserved = (
        torch.cuda.memory_reserved(0)
        / (1024 ** 3)
    )


    print(
        f"GPU             : {gpu_name}"
    )

    print(
        f"Total VRAM      : {total_vram:.2f} GB"
    )

    print(
        f"Allocated VRAM  : {allocated:.3f} GB"
    )

    print(
        f"Reserved VRAM   : {reserved:.3f} GB"
    )

else:

    print(
        "CUDA is not available."
    )

GPU MEMORY STATUS
GPU             : NVIDIA GeForce RTX 3050 Laptop GPU
Total VRAM      : 4.00 GB
Allocated VRAM  : 0.000 GB
Reserved VRAM   : 0.000 GB


In [20]:
# ============================================================
# SECTION 14 — FORWARD PASS TEST
# ============================================================

import numpy as np


print("=" * 70)
print("FORWARD PASS TEST")
print("=" * 70)


dummy_image = np.zeros(
    (IMGSZ, IMGSZ, 3),
    dtype=np.uint8
)


try:

    prediction = model.predict(

        source=dummy_image,

        imgsz=IMGSZ,

        device=DEVICE,

        verbose=False
    )

    print(
        "\n✓ Forward pass successful."
    )

except Exception as e:

    print(
        "\n✗ Forward pass failed."
    )

    print(
        "\nError:"
    )

    print(e)

    raise

FORWARD PASS TEST

✓ Forward pass successful.


In [21]:
# ============================================================
# SECTION 15 — FULL 100-EPOCH TRAINING
# ============================================================

print("=" * 70)
print("STARTING EXPERIMENT 6 — ECOBOTX-EDGE")
print("=" * 70)

print(
    f"\nEpochs     : {EPOCHS}"
)

print(
    f"Image size : {IMGSZ}"
)

print(
    f"Batch size : {BATCH}"
)

print(
    f"Device     : {DEVICE}"
)


# ------------------------------------------------------------
# CLEAR GPU MEMORY
# ------------------------------------------------------------

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ------------------------------------------------------------
# TRAINING START TIME
# ------------------------------------------------------------

training_start = time.time()


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

train_results = model.train(

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    data=DATA,

    # --------------------------------------------------------
    # Main training
    # --------------------------------------------------------

    epochs=100,

    imgsz=IMGSZ,

    batch=BATCH,

    device=DEVICE,

    project=PROJECT,

    name=NAME,

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer="SGD",

    lr0=0.01,

    lrf=0.01,

    momentum=0.937,

    weight_decay=0.0005,

    warmup_epochs=3.0,

    cos_lr=True,

    patience=30,

    seed=0,

    # --------------------------------------------------------
    # RTX 3050 optimization
    # --------------------------------------------------------

    amp=True,

    cache=True,

    workers=WORKERS,

    # --------------------------------------------------------
    # Save / plots
    # --------------------------------------------------------

    save=True,

    plots=True,

    # --------------------------------------------------------
    # Augmentation
    # --------------------------------------------------------

    hsv_h=0.015,

    hsv_s=0.7,

    hsv_v=0.4,

    degrees=5.0,

    translate=0.1,

    scale=0.5,

    fliplr=0.5,

    flipud=0.0,

    mosaic=1.0,

    close_mosaic=10,

    mixup=0.1,

    copy_paste=0.2
)


# ------------------------------------------------------------
# TRAINING TIME
# ------------------------------------------------------------

training_time = (
    time.time() - training_start
)


print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    f"\nTraining time: "
    f"{training_time / 3600:.2f} hours"
)

STARTING EXPERIMENT 6 — ECOBOTX-EDGE

Epochs     : 100
Image size : 416
Batch size : 8
Device     : 0
New https://pypi.org/project/ultralytics/8.4.130 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.126  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=G:\EcoBotX_YOLO\dataset.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=

In [23]:
# ============================================================
# SECTION 26 — LOAD EXPERIMENT 6 BEST MODEL
# ============================================================

from pathlib import Path
import torch
from ultralytics import YOLO

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

MODEL_PATH = Path(
    r"G:\EcoBotX_YOLO_training\experiment6_ecobotx_edge-2\weights\best.pt"
)

DATASET_YAML = r"G:\EcoBotX_YOLO\dataset.yaml"

# ------------------------------------------------------------
# CHECK MODEL
# ------------------------------------------------------------

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"best.pt not found:\n{MODEL_PATH}"
    )

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("=" * 70)
print("EXPERIMENT 6 — MODEL LOADING")
print("=" * 70)

print(f"\nModel : {MODEL_PATH}")
print(f"Device: {DEVICE}")

# ------------------------------------------------------------
# LOAD MODEL
# ------------------------------------------------------------

model = YOLO(
    str(MODEL_PATH)
)

print("\n✓ Experiment 6 best.pt loaded successfully.")

EXPERIMENT 6 — MODEL LOADING

Model : G:\EcoBotX_YOLO_training\experiment6_ecobotx_edge-2\weights\best.pt
Device: 0

✓ Experiment 6 best.pt loaded successfully.


In [24]:
# ============================================================
# SECTION 27 — TEST SET VALIDATION
# ============================================================

print("=" * 70)
print("EXPERIMENT 6 — TEST SET VALIDATION")
print("=" * 70)

print("\nRunning validation on:")
print("TEST split")

print("\nThis may take a few seconds...\n")

test_results = model.val(
    data=DATASET_YAML,
    split="test",
    imgsz=416,
    batch=8,
    device=DEVICE,
    workers=4,
    plots=True,
    verbose=True
)

print("\n✓ Test validation completed.")

EXPERIMENT 6 — TEST SET VALIDATION

Running validation on:
TEST split

This may take a few seconds...

Ultralytics 8.4.126  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
ecobotx_edge summary (fused): 147 layers, 1,714,856 parameters, 0 gradients, 5.0 GFLOPs
WARNING val: Slow image access detected (ping: 0.20.1 ms, read: 11.52.0 MB/s, size: 4.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning G:\EcoBotX_YOLO\labels\test.cache... 1115 images, 216 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1115/1115  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 140/140 19.5it/s 7.2s0.1s
                   all       1115        899      0.924      0.908      0.963       0.63
                BOTTLE        217        217       0.93      0.925      0.977      0.573
                   CAN        228  

In [26]:
# ============================================================
# EXPERIMENT 6 — CAN FAILURE ANALYSIS
# SECTION 7 — CAN-SPECIFIC PREDICTION ANALYSIS
# ============================================================

from pathlib import Path
import json
import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_PATH = Path(
    r"G:\EcoBotX_YOLO_training\experiment6_ecobotx_edge-2\weights\best.pt"
)

DATASET_ROOT = Path(
    r"G:\EcoBotX_YOLO"
)

TEST_IMAGES = DATASET_ROOT / "images" / "test"
TEST_LABELS = DATASET_ROOT / "labels" / "test"

OUTPUT_ROOT = Path(
    r"G:\EcoBotX_YOLO"
    r"\experiment6_can_failure_analysis"
    r"\section7_can_analysis"
)

MISSED_DIR = OUTPUT_ROOT / "missed_can"
WRONG_DIR = OUTPUT_ROOT / "wrong_class_can"
LOW_CONF_DIR = OUTPUT_ROOT / "low_confidence_can"

JSON_PATH = OUTPUT_ROOT / "section7_can_analysis.json"


# ============================================================
# MODEL CONFIGURATION
# ============================================================

CAN_ID = 1

CLASS_NAMES = {
    0: "BOTTLE",
    1: "CAN",
    2: "PAPER",
    3: "WRAPPER",
}

CONF_THRESHOLD = 0.50
IOU_MATCH_THRESHOLD = 0.50


# ============================================================
# DIRECTORY SETUP
# ============================================================

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

MISSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

WRONG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LOW_CONF_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# IoU FUNCTION
# ============================================================

def calculate_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_width = max(
        0,
        x2 - x1
    )

    intersection_height = max(
        0,
        y2 - y1
    )

    intersection = (
        intersection_width *
        intersection_height
    )

    area1 = max(
        0,
        box1[2] - box1[0]
    ) * max(
        0,
        box1[3] - box1[1]
    )

    area2 = max(
        0,
        box2[2] - box2[0]
    ) * max(
        0,
        box2[3] - box2[1]
    )

    union = area1 + area2 - intersection

    if union <= 0:

        return 0.0

    return intersection / union


# ============================================================
# YOLO LABEL → PIXEL BOX
# ============================================================

def yolo_to_xyxy(
    x_center,
    y_center,
    width,
    height,
    image_width,
    image_height
):

    x1 = (
        x_center -
        width / 2
    ) * image_width

    y1 = (
        y_center -
        height / 2
    ) * image_height

    x2 = (
        x_center +
        width / 2
    ) * image_width

    y2 = (
        y_center +
        height / 2
    ) * image_height

    return [
        x1,
        y1,
        x2,
        y2
    ]


# ============================================================
# LOAD CAN GROUND TRUTH
# ============================================================

def load_can_annotations():

    records = []

    image_files = sorted(
        list(TEST_IMAGES.glob("*.jpg")) +
        list(TEST_IMAGES.glob("*.jpeg")) +
        list(TEST_IMAGES.glob("*.png"))
    )

    for image_path in image_files:

        label_path = (
            TEST_LABELS /
            f"{image_path.stem}.txt"
        )

        if not label_path.exists():

            continue

        image = cv2.imread(
            str(image_path)
        )

        if image is None:

            continue

        image_height, image_width = (
            image.shape[:2]
        )

        with open(
            label_path,
            "r"
        ) as f:

            lines = f.readlines()

        for line in lines:

            parts = line.strip().split()

            if len(parts) != 5:

                continue

            class_id = int(parts[0])

            if class_id != CAN_ID:

                continue

            xc = float(parts[1])
            yc = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])

            box = yolo_to_xyxy(
                xc,
                yc,
                w,
                h,
                image_width,
                image_height
            )

            records.append({

                "image": str(image_path),

                "image_name":
                    image_path.name,

                "gt_box": box,

                "xc": xc,
                "yc": yc,
                "width": w,
                "height": h,

                "area": w * h,

                "aspect_ratio":
                    w / h
                    if h > 0
                    else 0,

            })

    return records


# ============================================================
# DRAW FAILURE IMAGE
# ============================================================

def save_failure_image(
    image_path,
    gt_box,
    prediction,
    output_path
):

    image = cv2.imread(
        str(image_path)
    )

    if image is None:

        return

    # Ground truth
    x1, y1, x2, y2 = map(
        int,
        gt_box
    )

    cv2.rectangle(
        image,
        (x1, y1),
        (x2, y2),
        (255, 255, 255),
        2
    )

    cv2.putText(
        image,
        "GT: CAN",
        (x1, max(20, y1 - 8)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    # Prediction
    if prediction is not None:

        px1, py1, px2, py2 = map(
            int,
            prediction["box"]
        )

        label = (
            prediction["class_name"] +
            f" {prediction['confidence']:.2f}"
        )

        cv2.rectangle(
            image,
            (px1, py1),
            (px2, py2),
            (180, 180, 180),
            2
        )

        cv2.putText(
            image,
            label,
            (px1, max(20, py1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (180, 180, 180),
            2
        )

    cv2.imwrite(
        str(output_path),
        image
    )


# ============================================================
# MAIN ANALYSIS
# ============================================================

def main():

    print("=" * 70)
    print(
        "EXPERIMENT 6 — CAN FAILURE ANALYSIS"
    )
    print(
        "SECTION 7 — CAN PREDICTION ANALYSIS"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # PATH CHECK
    # --------------------------------------------------------

    print("\n[1] Checking paths...")

    if not MODEL_PATH.exists():

        raise FileNotFoundError(
            f"Model not found:\n{MODEL_PATH}"
        )

    if not TEST_IMAGES.exists():

        raise FileNotFoundError(
            f"Test images not found:\n{TEST_IMAGES}"
        )

    if not TEST_LABELS.exists():

        raise FileNotFoundError(
            f"Test labels not found:\n{TEST_LABELS}"
        )

    print("[OK] Model found")
    print("[OK] Test images found")
    print("[OK] Test labels found")

    # --------------------------------------------------------
    # LOAD MODEL
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("LOADING EXPERIMENT 6 MODEL")
    print("=" * 70)

    model = YOLO(
        str(MODEL_PATH)
    )

    print("[OK] Model loaded")

    print("\nModel classes:")

    for k, v in model.names.items():

        print(
            f"  {k}: {v}"
        )

    # --------------------------------------------------------
    # LOAD GROUND TRUTH
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("LOADING CAN GROUND TRUTH")
    print("=" * 70)

    records = load_can_annotations()

    print(
        f"CAN annotations: {len(records)}"
    )

    # --------------------------------------------------------
    # RUN MODEL
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("RUNNING EXPERIMENT 6")
    print("=" * 70)

    image_cache = {}

    for record in tqdm(
        records,
        desc="Analyzing CAN images"
    ):

        image_path = record["image"]

        if image_path not in image_cache:

            result = model.predict(
                source=image_path,
                conf=0.20,
                verbose=False
            )[0]

            image_cache[
                image_path
            ] = result

        result = image_cache[
            image_path
        ]

        gt_box = record["gt_box"]

        predictions = []

        if result.boxes is not None:

            for i in range(
                len(result.boxes)
            ):

                box = result.boxes.xyxy[
                    i
                ].cpu().numpy()

                confidence = float(
                    result.boxes.conf[
                        i
                    ].cpu().item()
                )

                class_id = int(
                    result.boxes.cls[
                        i
                    ].cpu().item()
                )

                iou = calculate_iou(
                    gt_box,
                    box
                )

                predictions.append({

                    "box":
                        box.tolist(),

                    "confidence":
                        confidence,

                    "class_id":
                        class_id,

                    "class_name":
                        CLASS_NAMES.get(
                            class_id,
                            str(class_id)
                        ),

                    "iou":
                        iou,
                })

        # ----------------------------------------------------
        # Find best localization
        # ----------------------------------------------------

        if predictions:

            best = max(
                predictions,
                key=lambda x: x["iou"]
            )

        else:

            best = None

        # ----------------------------------------------------
        # Classification logic
        # ----------------------------------------------------

        can_predictions = [

            p for p in predictions

            if p["class_id"] == CAN_ID
        ]

        if can_predictions:

            best_can = max(
                can_predictions,
                key=lambda x: x["iou"]
            )

        else:

            best_can = None

        # ----------------------------------------------------
        # Correct CAN
        # ----------------------------------------------------

        if (
            best_can is not None
            and best_can["iou"]
                >= IOU_MATCH_THRESHOLD
        ):

            record["status"] = (
                "correct_high_conf"
                if best_can["confidence"]
                >= CONF_THRESHOLD
                else
                "correct_low_conf"
            )

            record["best_iou"] = (
                best_can["iou"]
            )

            record["confidence"] = (
                best_can["confidence"]
            )

            record["predicted_class"] = "CAN"

        # ----------------------------------------------------
        # Wrong class
        # ----------------------------------------------------

        elif (
            best is not None
            and best["iou"]
                >= IOU_MATCH_THRESHOLD
            and best["class_id"]
                != CAN_ID
        ):

            record["status"] = (
                "wrong_class"
            )

            record["best_iou"] = (
                best["iou"]
            )

            record["confidence"] = (
                best["confidence"]
            )

            record["predicted_class"] = (
                best["class_name"]
            )

            filename = (
                image_path
            )

            output_name = (
                Path(filename).stem +
                "_wrong.jpg"
            )

            save_failure_image(
                image_path,
                gt_box,
                best,
                WRONG_DIR / output_name
            )

        # ----------------------------------------------------
        # Missed
        # ----------------------------------------------------

        else:

            record["status"] = "missed"

            record["best_iou"] = (
                best["iou"]
                if best is not None
                else 0.0
            )

            record["confidence"] = (
                best["confidence"]
                if best is not None
                else 0.0
            )

            record["predicted_class"] = (
                best["class_name"]
                if best is not None
                else None
            )

            output_name = (
                Path(image_path).stem +
                "_missed.jpg"
            )

            save_failure_image(
                image_path,
                gt_box,
                best,
                MISSED_DIR / output_name
            )

        # ----------------------------------------------------
        # Low confidence output
        # ----------------------------------------------------

        if (
            record["status"]
            == "correct_low_conf"
        ):

            output_name = (
                Path(image_path).stem +
                "_low_conf.jpg"
            )

            save_failure_image(
                image_path,
                gt_box,
                best_can,
                LOW_CONF_DIR / output_name
            )

    # ========================================================
    # SUMMARY
    # ========================================================

    total = len(records)

    correct_high = sum(
        r["status"] ==
        "correct_high_conf"
        for r in records
    )

    correct_low = sum(
        r["status"] ==
        "correct_low_conf"
        for r in records
    )

    missed = sum(
        r["status"] ==
        "missed"
        for r in records
    )

    wrong = sum(
        r["status"] ==
        "wrong_class"
        for r in records
    )

    successful = (
        correct_high +
        correct_low
    )

    recall = (
        successful / total
        if total > 0
        else 0
    )

    print("\n" + "=" * 70)
    print("SECTION 7 — CAN PREDICTION RESULTS")
    print("=" * 70)

    print(
        f"Total CAN ground-truth boxes : {total}"
    )

    print(
        f"Correct CAN — high conf.    : "
        f"{correct_high}"
    )

    print(
        f"Correct CAN — low conf.     : "
        f"{correct_low}"
    )

    print(
        f"Missed CAN                  : "
        f"{missed}"
    )

    print(
        f"Wrong-class CAN             : "
        f"{wrong}"
    )

    print(
        f"Successful CAN detections   : "
        f"{successful}"
    )

    print(
        f"CAN Recall                  : "
        f"{recall:.4f} ({recall*100:.2f}%)"
    )

    # ========================================================
    # SAVE JSON
    # ========================================================

    output = {

        "experiment":
            "Experiment 6",

        "section":
            "Section 7",

        "model":
            str(MODEL_PATH),

        "total_can":
            total,

        "correct_high_confidence":
            correct_high,

        "correct_low_confidence":
            correct_low,

        "missed":
            missed,

        "wrong_class":
            wrong,

        "successful_detection":
            successful,

        "can_recall":
            recall,

        "records":
            records,
    }

    with open(
        JSON_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            output,
            f,
            indent=2
        )

    print()
    print(
        f"[OK] JSON saved:\n{JSON_PATH}"
    )

    print("\n" + "=" * 70)
    print(
        "SECTION 7 — COMPLETED"
    )
    print("=" * 70)


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()

EXPERIMENT 6 — CAN FAILURE ANALYSIS
SECTION 7 — CAN PREDICTION ANALYSIS

[1] Checking paths...
[OK] Model found
[OK] Test images found
[OK] Test labels found

LOADING EXPERIMENT 6 MODEL
[OK] Model loaded

Model classes:
  0: BOTTLE
  1: CAN
  2: PAPER
  3: WRAPPER

LOADING CAN GROUND TRUTH
CAN annotations: 228

RUNNING EXPERIMENT 6


Analyzing CAN images: 100%|██████████| 228/228 [00:04<00:00, 53.58it/s]



SECTION 7 — CAN PREDICTION RESULTS
Total CAN ground-truth boxes : 228
Correct CAN — high conf.    : 186
Correct CAN — low conf.     : 30
Missed CAN                  : 11
Wrong-class CAN             : 1
Successful CAN detections   : 216
CAN Recall                  : 0.9474 (94.74%)


TypeError: Object of type float32 is not JSON serializable